##### Import the libraries

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import holidays
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score)
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import joblib
import os
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

##### Load the datasets

In [51]:
import pandas as pd

df = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Advanced Models\Predictions\sarimax_predictions.csv")

print(df.columns.tolist())

df.head()

['Unit', 'Date', 'Actual', 'Predicted']


,Unit,Date,Actual,Predicted
0,HHN-BIR-01_ICU,2025-12-02,7.958333,8.083947
1,HHN-BIR-01_ICU,2025-12-03,8.500000,7.735496
2,HHN-BIR-01_ICU,2025-12-04,7.208333,7.542131
3,HHN-BIR-01_ICU,2025-12-05,5.541667,7.248901
4,HHN-BIR-01_ICU,2025-12-06,5.250000,7.000670


In [36]:
model_folder = r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Best Trained Models"
data_path = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Data\Cleaned\bed_inventory_cleaned.csv")

In [37]:
data_path.head()

,datetime,hospital_id,ward,bed_type,total_beds,staffed_beds,occupied_beds,closed_beds,occupancy_rate,bedding_year,bedding_month,bedding_quarter,bedding_weekday,bedding_hour
0,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,0
1,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,1
2,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,2
3,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,3
4,2024-01-01,HHN-BIR-01,Cardiology Ward,Standard,42,42,0,0,0.0,2024,January,1,Monday,4


In [38]:

data_path["datetime"] = pd.to_datetime(data_path["datetime"])

In [39]:
model_files = os.listdir(model_folder)

model_files[:10]

['holt_HHN-BIR-01_General Medicine Ward A.pkl',
 'holt_HHN-BIR-01_ICU.pkl',
 'holt_HHN-BIR-01_Oncology Ward.pkl',
 'holt_HHN-BIR-01_Orthopaedics Ward A.pkl',
 'holt_HHN-EDI-01_Cardiology Ward.pkl',
 'holt_HHN-EDI-01_General Medicine Ward A.pkl',
 'holt_HHN-EDI-01_General Medicine Ward B.pkl',
 'holt_HHN-EDI-01_ICU.pkl',
 'holt_HHN-EDI-01_Oncology Ward.pkl',
 'holt_HHN-EDI-01_Orthopaedics Ward A.pkl']

##### Create function to load model

In [41]:
def load_model(model_file):

    path = os.path.join(model_folder, model_file)
    model = joblib.load(path)
    return model

model = load_model(model_files[0])
model

##### Create scenerio function

In [42]:
def apply_scenario(forecast, increase):
    stressed_forecast = forecast * (1 + increase)
    return stressed_forecast

##### Define scenerios

In [43]:
scenarios = {"Baseline": 0, "Flu Surge +20%": 0.20, "Discharge Delay +15%": 0.15, "Extreme Stress +35%": 0.35}

In [48]:
# Store the results
scenario_results = []

# Loop through all saved models
for model_file in model_files:
    print("Processing:", model_file)

    # Load model
    model = load_model(model_file)

    # Extract unit name
    unit = (
        model_file
        .replace(".pkl","")
        .replace("holt_","")
        .replace("sarima_","")
        .replace("sarimax_","")
        .replace("xgboost_","")
    )

    # Select historical data for unit
    unit_data = data_path[data_path["ward"] == ward].copy()

    if len(unit_data) == 0:
        print("No data found for", ward)
        continue

    # Sort data
    unit_data = unit_data.sort_values("datetime")

    # Forecast horizon
    forecast_days = 7



    # Generate forecast

    try:

        forecast = model.forecast(
            forecast_days
        )


    except:


        forecast = model.predict(
            forecast_days
        )



    # Apply scenarios

    for scenario, increase in scenarios.items():


        stressed_forecast = apply_scenario(
            forecast,
            increase
        )


        for i, value in enumerate(stressed_forecast):


            scenario_results.append({

                "unit": unit,

                "model": model_file,

                "scenario": scenario,

                "day": i+1,

                "forecast_demand": value

            })


Processing: holt_HHN-BIR-01_General Medicine Ward A.pkl
Processing: holt_HHN-BIR-01_ICU.pkl
Processing: holt_HHN-BIR-01_Oncology Ward.pkl
Processing: holt_HHN-BIR-01_Orthopaedics Ward A.pkl
Processing: holt_HHN-EDI-01_Cardiology Ward.pkl
Processing: holt_HHN-EDI-01_General Medicine Ward A.pkl
Processing: holt_HHN-EDI-01_General Medicine Ward B.pkl
Processing: holt_HHN-EDI-01_ICU.pkl
Processing: holt_HHN-EDI-01_Oncology Ward.pkl
Processing: holt_HHN-EDI-01_Orthopaedics Ward A.pkl
Processing: holt_HHN-EDI-01_Orthopaedics Ward B.pkl
Processing: holt_HHN-LON-01_ICU.pkl
Processing: holt_HHN-LON-01_Orthopaedics Ward B.pkl
Processing: holt_HHN-LON-02_General Medicine Ward A.pkl
Processing: holt_HHN-LON-02_Orthopaedics Ward A.pkl
Processing: holt_HHN-MAN-01_Cardiology Ward.pkl
Processing: holt_HHN-MAN-01_General Medicine Ward A.pkl
Processing: holt_HHN-MAN-01_General Medicine Ward B.pkl
Processing: holt_HHN-MAN-01_ICU.pkl
Processing: holt_HHN-MAN-01_Orthopaedics Ward A.pkl
Processing: sarimax_

ValueError: data did not contain feature names, but the following fields are expected: day_of_week, month, quarter, week_of_year, is_weekend, is_public_holiday, lag_7, rolling_mean_7, avg_los